## Task 1: Load and Inspect the Data
Create the DataFrame using the provided code, then:

Display basic information using df.info()
Count missing values in each column using df.isna().sum()

In [30]:
import pandas as pd
import numpy as np

data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home', 
                         'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing', 
                         'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics', 
                         'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780, 
                     560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5, 
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41, 
                     33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card', 
                       'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card', 
                       'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card', 
                       'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)
print(df.info())
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    20 non-null     int64         
 1   date              20 non-null     datetime64[us]
 2   region            17 non-null     str           
 3   product_category  18 non-null     str           
 4   sales_amount      16 non-null     float64       
 5   quantity          17 non-null     float64       
 6   customer_age      16 non-null     float64       
 7   payment_method    17 non-null     str           
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 1.4 KB
None
transaction_id      0
date                0
region              3
product_category    2
sales_amount        4
quantity            3
customer_age        4
payment_method      3
dtype: int64


## Task 2: Handle Missing Values
Clean the dataset using appropriate strategies:

1. Region & Product_category: Fill with mode (most frequent value)
2. Sales_amount: Fill with median
3. Quantity: Fill using forward fill (ffill)
4. Customer_age: Fill with mean (rounded to integer)
5. Payment_method: Drop rows where payment_method is missing
6. Verify no missing values remain using df.isna().sum()

In [37]:
#.mode() will return series, so we are doing .mode()[0] to get the first value 
df = df.fillna({'region':df['region'].mode()[0],
    'product_category':df['product_category'].mode()[0],
    'sales_amount': df['sales_amount'].median(),'customer_age':round(df['customer_age'].mean())})
df['quantity'] = df['quantity'].ffill()
df = df.dropna(subset=['payment_method'])
print(df.isna().sum())
df

transaction_id      0
date                0
region              0
product_category    0
sales_amount        0
quantity            0
customer_age        0
payment_method      0
dtype: int64


,transaction_id,date,region,product_category,sales_amount,quantity,customer_age,payment_method
0,1,2024-10-01,North,Electronics,1200.0,2.0,25.0,Credit Card
1,2,2024-10-02,South,Clothing,450.0,5.0,34.0,UPI
2,3,2024-10-03,East,Clothing,890.0,3.0,37.0,Cash
3,4,2024-10-04,West,Books,725.0,1.0,45.0,Debit Card
4,5,2024-10-05,North,Electronics,1500.0,1.0,29.0,Credit Card
5,6,2024-10-06,North,Home,670.0,4.0,37.0,UPI
6,7,2024-10-07,South,Clothing,725.0,2.0,38.0,Cash
8,9,2024-10-09,East,Electronics,2100.0,1.0,27.0,Credit Card
9,10,2024-10-10,West,Clothing,780.0,5.0,41.0,UPI
10,11,2024-10-11,North,Home,560.0,3.0,33.0,Debit Card


## Task 3: GroupBy Analysis
Perform the following aggregations:

1. Calculate total sales by region
2. Calculate average sales by product_category
3. Group by both region and product_category, calculate total sales and quantity, then use reset_index() to flatten
4. Display top 3 region-product combinations by sales

In [60]:
print("-------------------Total sales per region----------------")
print(df.groupby('region')['sales_amount'].sum())
print("-------------------Avg sales per product category----------------")
print(df.groupby('product_category')['sales_amount'].mean())
print("------------------Total sales and quantity----------------")
print(df.groupby(['region','product_category']).agg({'sales_amount':'sum','quantity':'sum'}).reset_index())
print("------------TOP 3 PRODUCTS------------------")
print(df.groupby(['region','product_category'])['sales_amount'].sum().reset_index().sort_values(by='sales_amount',ascending=False).head(3))

-------------------Total sales per region----------------
region
East     3790.0
North    6460.0
South    1900.0
West     2230.0
Name: sales_amount, dtype: float64
-------------------Avg sales per product category----------------
product_category
Books           508.333333
Clothing        680.000000
Electronics    1381.250000
Home            812.500000
Name: sales_amount, dtype: float64
------------------Total sales and quantity----------------
  region product_category  sales_amount  quantity
0   East            Books         800.0       4.0
1   East         Clothing         890.0       3.0
2   East      Electronics        2100.0       1.0
3  North         Clothing         510.0       3.0
4  North      Electronics        2700.0       3.0
5  North             Home        3250.0      12.0
6  South         Clothing        1900.0       9.0
7   West            Books         725.0       1.0
8   West         Clothing         780.0       5.0
9   West      Electronics         725.0       1.0
-

## Task 4: Custom Aggregation
Create a function sales_range() that returns max - min of sales
1. Apply it to find sales range for each region
2. Use .agg() to calculate multiple metrics by region:
 1. sales_amount: sum, mean, max
 2. quantity: sum, min

In [64]:
def sales_range(x):
    return x.max() - x.min()

print('-------------SALES RANGE PER REGION----------------')
print(df.groupby('region').agg({'sales_amount':sales_range}))    

print('----------------MULTIPLE METRICS-------------------')
print(df.groupby('region')
    .agg({'sales_amount':['sum','mean','max'],
         'quantity':['sum','min']}))


-------------SALES RANGE PER REGION----------------
        sales_amount
region              
East          1720.0
North          990.0
South          275.0
West            55.0
----------------MULTIPLE METRICS-------------------
       sales_amount                     quantity     
                sum        mean     max      sum  min
region                                               
East         3790.0  947.500000  2100.0      8.0  1.0
North        6460.0  922.857143  1500.0     18.0  1.0
South        1900.0  633.333333   725.0      9.0  2.0
West         2230.0  743.333333   780.0      7.0  1.0
